# Silver — Cost Entries

Lê Delta Bronze, aplica qualidade e enriquece com `team_name`.

**Transformações:**
- Filtra `cost_usd >= 0`, nulos em campos obrigatórios
- Deduplica por `(team_id, usage_date, resource_name)`
- Normaliza `environment` → `prod` / `staging` / `dev`
- Join com `silver/teams` para adicionar `team_name`

In [ ]:
# parameters
execution_date = "2025-06-15"
year = 2025
month = 6
day = 15


In [ ]:
import sys
import os

notebooks_root = next(
    (p for p in ["/opt/airflow/notebooks", "/opt/spark/notebooks"]
     if __import__("os").path.exists(p)),
    "/opt/spark/notebooks",
)
if notebooks_root not in sys.path:
    sys.path.insert(0, notebooks_root)

from utils.spark_session import create_spark_session, postgres_jdbc_url, postgres_props
from pyspark.sql import functions as F
from pyspark.sql.types import (
    IntegerType, StringType, DecimalType, TimestampType, DateType
)


In [ ]:
spark = create_spark_session("silver_cost_entries")

# Processa TODOS os meses presentes no bronze (full reprocess)
df_entries = spark.read.format("delta").load("s3a://datalake/bronze/cost_entries/")

df_teams = (
    spark.read.format("delta").load("s3a://datalake/silver/teams/")
    .select(F.col("id").alias("team_id"), F.col("name").alias("team_name"))
)

df_silver = (
    df_entries
    # Qualidade
    .filter(F.col("cost_usd") >= 0)
    .filter(F.col("team_id").isNotNull())
    .filter(F.col("usage_date").isNotNull())
    .filter(F.col("resource_name").isNotNull())
    # Deduplicação por chave natural
    .dropDuplicates(["team_id", "usage_date", "resource_name"])
    # Normalização do ambiente
    .withColumn(
        "environment",
        F.when(F.lower(F.col("environment")).isin("prod", "production"), "prod")
         .when(F.lower(F.col("environment")).isin("staging", "stg"), "staging")
         .when(F.lower(F.col("environment")).isin("dev", "development"), "dev")
         .otherwise(F.lower(F.col("environment"))),
    )
    # Enriquecimento com nome do time
    .join(df_teams, on="team_id", how="left")
    .drop("_ingested_at", "_source_layer")
    .withColumn("_processed_at", F.current_timestamp())
)

output_path = "s3a://datalake/silver/cost_entries/"
(
    df_silver.write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .save(output_path)
)

count = df_silver.count()
print(f"[silver_cost_entries] {count} registros → {output_path}")
df_silver.groupBy("environment", "provider").count().orderBy("environment", "provider").show()
spark.stop()
